# పాఠం 13 - ఏజెంట్ మెమరీ


## సెటప్

ఈ నోట్‌బుక్ **Microsoft Agent Framework** (MAF) ఉపయోగించి **పర్మనెంట్ మెమొరీ**తో ట్రావెల్ బుకింగ్ ఏజెంట్ను ఎలా సృష్టించాలో చూపిస్తుంది.

విభిన్న రకాల ఏజెంట్ మెమొరీ — వర్కింగ్, షార్ట్-టర్మ్, మరియు లాంగ్-టర్మ్ — ఏజెంట్ సంభాషణలలో సమాచారం ఎలా నిలిపి ఉంచి ఉపయోగిస్తుందో మీరు తెలుసుకుంటారు.

**అవసరాలు:**
- వోసిపోయిన చాట్ మోడల్ (ఉదాహరణకు `gpt-4o-mini`)తో ఒక Azure AI Foundry ప్రాజెక్టు.
- Azure CLI లో లాగిన్ అయి ఉండాలి — మీ టెర్మినల్‌లో `az login` రన్ చేయండి.
- `AZURE_AI_PROJECT_ENDPOINT` — మీ Azure AI Foundry ప్రాజెక్టు ఎండ్పాయింట్.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీరు డిప్లాయ్ చేసిన మోడల్ పేరు.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ఏజెంట్ మెమరీ రకాలు

AI ఏజెంట్లు వేర్వేరు రకాల మెమరీలను ఉపయోగించవచ్చు, ప్రతి ఒక్కదీ ఒక ప్రత్యేక యూత్ కోసం ఉంటుంది:

### వర్కింగ్ మెమరీ  
సంభాషణ థ్రెడ్ స్వయంగా — ఒక్క సెషన్‌లో మార్పిడీ అయిన సందేశాలు. ఏజెంట్ ఒకే థ్రెడ్‌లో ముందర సందేశాలను సూచించి సుసంపత్తిని నిలుపుకోవచ్చు. MAFలో ఇది **`agent.create_session()`** ద్వారా సృష్టించబడుతుంది, ఇది ఒక `AgentSession`ని తిరిగి ఇస్తుంది.

### షార్ట్-టర్మ్ మెమరీ  
ఒక టాస్క్ లేదా సెషన్ కాలంలో నిలిచిపోయే సమాచారాన్ని సూచిస్తుంది కానీ శాశ్వతంగా నిల్వ చేయబడదు. ఉదాహరణకు, ఏజెంట్ బహుళ-టర్న్ ప్రణాళిక సంభాషణలో నిజాలు సేకరించి అవి ఉపయోగించి తుది ప్రయాణ ప్రణాళికను తయారు చేయవచ్చు.

### లాంగ్-టర్మ్ మెమరీ  
**సెషన్ల మధ్య** నిలిచే అభిరుచులు మరియు నిజాలు. తిరిగి వచ్చిన వినియోగదారు తన ఆహార పరిమితులు లేదా ప్రయాణ శైలి మళ్లీ చెప్పాల్సిన అవసరం ఉండకూడదు. దీర్ఘకాలిక మెమరీ సాధారణంగా బయటి నిల్వ ద్వారా మద్దతు పొందుతుంది — డేటాబేస్, ఫైల్ లేదా వెక్టర్ సూచిక — మరియు ఏజెంట్‌కు టూల్స్ ద్వారా అందించబడుతుంది.


## సెషన్లతో పనిచేసే మెమరీ

అతి సాధారణమైన మెమరీ రూపం సంభాషణ సెషన్. మీరు అదే సెషన్ ( `agent.create_session()` ద్వారా సృష్టించబడింది) ను వరుసగా `agent.run()` కాల్స్ కు పంపినప్పుడు, ఏజెంట్ ఆ సంభాషణ యొక్క మొత్తం చరిత్రను చూడగలదు మరియు మొదటి వివరాలను గుర్తు చేసుకోవచ్చు.

వెళ్ళే ఏజెంట్‌ను సృష్టించి పని చేసే మెమరీని చూపించుకుందాం.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ఏజెంట్ బడ్జెట్‌ను సరైన రీతిలో గుర్తు చేసుకుంది ఎందుకంటే రెండు సందేశాలు ఒకే సెషన్‌ను పంచుకుంటున్నాయి. ఇది **పనిచేసే జ్ఞాపకం** — ఇది సెషన్ జీవితకాలం అంతే ఉంటుంది.

### కొత్త థ్రెడ్ తో ఏమి జరుగుతుంది?

మనము ఒక **కొత్త** సెషన్‌ను సృష్టిస్తే, ఏజెంట్‌కు గత సంభాషణ గురించి ఎలాంటి జ్ఞాపకం ఉండదు:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## దీర్ఘకాలిక జ్ఞాపకం నమూనా

ఇంటర్వ్యూలకు **మొత్తంగా** వినియోగదారుల ఇష్టాలను గుర్తుంచుకునేందుకు, సంభాషణ థ్రెడ్ వెలుపల జీవించే ఒక స్థిరమైన నిల్వ అవసరం. ఏజెంట్ ఈ నిల్వను **టూల్స్** ద్వారా యాక్సెస్ చేస్తుంది — సమాచారం సేవ్ చేయడానికి మరియు తీసుకోడానికి ఫంక్షన్లు.

క్రింద మేము ఒక సాదారణ ఇన్-మెమరీ ఇష్టాల నిల్వను అమలు చేస్తున్నాం (ఉత్పత్తిలో మీరు దీన్ని డేటాబేస్ లేదా వెక్టర్ సూచికతో మద్దతు ఇస్తారు) మరియు దాన్ని ఏజెంట్ ఉపయోగించగలటువంటి టూల్స్‌గా చూపిస్తున్నాం.

### నిర్మాణం
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### సందర్భం 1 — మొదటి సారి వినియోగదారుడు వార్షికోత్సవ ప్రయాణం బుక్ చేసుకుంటున్నారు

సారా మొదటి సారి సందర్శించటం జరిగింది. ఏజెంట్ ఆమె ఇష్టాలను టూల్స్ ద్వారా నిల్వ చేసి వాటిని ఉపయోగించి హోటల్స్ సిఫార్సు చేయాలి.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### సన్నివేశం 2 — సారా కొన్ని వారాల తరువాత తిరిగి వస్తుంది

సారా ఒక **తొలి సారి ప్రయోగం** ప్రారంభిస్తుంది (కొత్త సెషన్‌ను అనుకరించడం). వర్కింగ్ మెమోరి ఖాళీగా ఉంటుంది, కానీ దీర్ఘకాలిక ప్రాధాన్యత నిల్వలో ఆమె సమాచారం ఇంకా ఉంటుంది. ఏజెంట్ దాన్ని తీసుకుని వ్యక్తిగత సిఫారసులను సృష్టించడానికి ఉపయోగించాలి.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## సారాంశం

ఈ పాఠంలో మీరు మూడు రకాల ఏజెంట్ మెమరీలను మరియు వాటిని మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్‌తో ఎలా అమలు చేయాలో నేర్చుకున్నారు:

| మెమరీ రకం | MAF మెకానిజం | జీవకాలం |
|---|---|---|
| **వర్కింగ్** | `agent.create_session()` | ఒక్క సంభాషణ |
| **షార్ట్-టర్మ్** | థ్రెడులో కూడిన కాన్టెక్స్ట్ | ఒక్క టాస్క్ / సెషన్ |
| **లాంగ్-టర్మ్** | `@tool` ఫంక్షన్ల ద్వారా యాక్సెస్ అయ్యే బాహ్య నిల్వ | సెషన్ల మధ్య |

### ముఖ్య అంశాలు
1. **`agent.create_session()`** వర్కింగ్ మెమరీని అందిస్తుంది — ఏజెంట్ సెషన్‌లో పూర్తి సంభాషణ చారిత్రాన్ని చూస్తుంది.
2. **కొత్త సెషన్లు కాన్టెక్స్ట్‌ను కోల్పోతాయి** — లాంగ్-టర్మ్ మెమరీ లేకపోతే ఏజెంట్ గత సంభాషణలను గుర్తుబట్టలేకపోతుంది.
3. **`@tool` ఫంక్షన్లు** మధ్యంతరాన్ని కలిగిస్తాయి — అవి ఏజెంట్‌కు సుస్థిర నిల్వలో సమాచారాన్ని సేవ్ చేసి తిరిగి తీసుకోవడానికి వీలు కల్పిస్తాయి.
4. **వ్యక్తిగతీకరణ కాలక్రమేణా మెరుగుపడుతుంది** — ఎక్కువ ప్రిఫరెన్సులు నిల్వ చేస్తే, ఏజెంట్ సూచనలు మరింత మెరుగ్గా ఉంటాయి.

### వాస్తవ ప్రపంచ లో ఉపయోగకరతలు
- **కస్టమర్ సర్వీస్**: కస్టమర్ చరిత్ర మరియు ప్రాధాన్యతలను గుర్తుంచుట
- **వ్యక్తిగత సహాయకులు**: రోజులు లేదా వారాల పాటు కాన్టెక్స్ట్ నిల్వ చేయడం
- **ఆరోగ్యం**: రోగి సమాచారం మరియు ప్రాధాన్యతలను ట్రాక్ చేయడం
- **ఈ-కామర్స్**: చరిత్ర ఆధారంగా వ్యక్తిగత షాపింగ్

### తదుపరి దశలు
- ఇన్-మెమరీ డిక్ట్‌ను డేటాబేస్ లేదా వెక్టర్ స్టోర్ (ఉదా: Azure AI Search) తో మార్చండి
- స‌మ‌యానికి సంబంధించి సమాచారానికి మెమరీ కాలపరిమితి జోడించండి
- పంచుకున్న మెమరీతో బహుళ ఏజెంట్ వ్యవస్థలను నిర్మించండి
- జ్ఞాన-గ్రాఫ్ ఆధారిత మెమరీ కోసం Cognee నోటుబుక్‌ను అన్వేషించండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ఇతరాయం**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించాము. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా లోపాలు ఉండే అవకాశం ఉంటుంది. మొదటి భాషలో ఉన్న అసలు పత్రాన్ని అధికారిక మూలంగా పరిగణించాలి. కీలక సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదం సాధించడాన్ని మేము సిఫారసు చేస్తున్నాము. ఈ అనువాదం వాడుకున్న కారణంగా ఏర్పడిన ఏదైనా అపవాదాలు లేదా తప్పు అర్థాలపై మేము బాధ్యులు కేమి కాదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
